# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [ ]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
import time
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

In [ ]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")
print("NUMBER OF FILES =", len(g))

for f in sorted(g):
    print(f)

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    print("PROCESSING:", file)

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()

    if location.lower() == "online":
        location = None
        continue

    print("TITLE:", title)
    print("LOCATION:", location)

    talk_type = data.get('type', 'Other')
    #description = f"{title}<br />{venue}; {location}"
    description = f"[{talk_type}]<br />{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        #location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        #print(description, location_dict[description])
        result = geocoder.geocode(location, timeout=TIMEOUT)
        print("RESULT:", result)
        location_dict[description] = result
        time.sleep(1)
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

In [ ]:
#ajouté pour modifier map.htlm 
from pathlib import Path

mapfile = Path("talkmap/map.html")

html = mapfile.read_text(encoding="utf-8")

# Ajouter la fonction markerColor avant la boucle
html = html.replace(
    "for (var i = 0; i < addressPoints.length; i++) {",
    """
function markerColor(title) {

    if(title.includes("[Seminar]"))
        return "purple";

    if(title.includes("[Poster]"))
        return "blue";

    if(title.includes("[Invited talk]"))
        return "red";

    if(title.includes("[Invited review]"))
        return "darkred";

    if(title.includes("[Invited lecture]"))
        return "darkred";

    if(title.includes("[Oral contribution]"))
        return "green";

    return "orange";
}

for (var i = 0; i < addressPoints.length; i++) {
"""
)

# Remplacer le marqueur standard
html = html.replace(
    "var marker = L.marker(new L.LatLng(a[1], a[2]), { title: title });",
    """
var iconColor = markerColor(title);

var marker = L.marker(
    new L.LatLng(a[1], a[2]),
    {
        title: title,
        icon: new L.Icon({
            iconUrl:
                'https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-' +
                iconColor +
                '.png',
            shadowUrl:
                'https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png',
            iconSize: [25, 41],
            iconAnchor: [12, 41],
            popupAnchor: [1, -34],
            shadowSize: [41, 41]
        })
    }
);
"""
)

# Ajouter la légende
html = html.replace(
    "map.zoomIn();",
    """
map.zoomIn();

var legend = L.control({position: 'bottomright'});

legend.onAdd = function () {

    var div = L.DomUtil.create('div', 'info legend');

    div.style.backgroundColor = 'white';
    div.style.padding = '10px';
    div.style.border = '1px solid #999';

    div.innerHTML =
        '<b>Talk type</b><br>' +
        '<i style="background:red;width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Invited talk<br>' +
        '<i style="background:darkred;width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Invited review / lecture<br>' +
        '<i style="background:green;width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Oral contribution<br>' +
        '<i style="background:blue;width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Poster<br>' +
        '<i style="background:purple;width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Seminar';

    return div;
};

legend.addTo(map);
"""
)

mapfile.write_text(html, encoding="utf-8")

print("Custom map styling applied.")